In [ ]:
import Pkg
for pkg in ["CSV", "DataFrames", "GLMakie", "Colors"]
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

using CSV, DataFrames, GLMakie, Colors

# Frame transform helpers mirroring Supervisor.compute_task_frames().
function mat_vec_mul_row(v::NTuple{3, Float64}, m::AbstractMatrix{<:Real})
    return (
        v[1] * m[1, 1] + v[2] * m[2, 1] + v[3] * m[3, 1],
        v[1] * m[1, 2] + v[2] * m[2, 2] + v[3] * m[3, 2],
        v[1] * m[1, 3] + v[2] * m[2, 3] + v[3] * m[3, 3],
    )
end

function base_to_global_task_xyz(base_xyz::NTuple{3, Float64}, arm::Symbol)
    dy_t = 0.225 / 2 + 0.540 / 2
    dz_t = -0.753

    if arm == :left
        dx_t = 0.090 / 2 + 0.010 + 0.110
        r_task_to_base = Float64[
            0.707 0.0 -0.707;
            0.0 -1.0 0.0;
            -0.707 0.0 -0.707;
        ]
    elseif arm == :right
        dx_t = -(0.090 / 2 + 0.010 + 0.110)
        r_task_to_base = Float64[
            0.707 0.0 0.707;
            0.0 -1.0 0.0;
            0.707 0.0 -0.707;
        ]
    else
        error("arm must be :left or :right")
    end

    trans_base_to_task = mat_vec_mul_row((dx_t, dy_t, dz_t), r_task_to_base)
    p_rel = (
        base_xyz[1] - trans_base_to_task[1],
        base_xyz[2] - trans_base_to_task[2],
        base_xyz[3] - trans_base_to_task[3],
    )
    return mat_vec_mul_row(p_rel, transpose(r_task_to_base))
end

# Load robot log CSV (same folder as this notebook)
df = CSV.read("robot_data_right.csv", DataFrame)
source_arm = :right

# Arm/base-frame XYZ
x_arm = Float32.(df.actual_TCP_pose_0)
y_arm = Float32.(df.actual_TCP_pose_1)
z_arm = Float32.(df.actual_TCP_pose_2)

# Global task-frame XYZ (for plotting)
global_xyz = [
    base_to_global_task_xyz((Float64(x_arm[i]), Float64(y_arm[i]), Float64(z_arm[i])), source_arm)
    for i in eachindex(x_arm)
]
x = Float32[getindex(p, 1) for p in global_xyz]
y = Float32[getindex(p, 2) for p in global_xyz]
z = Float32[getindex(p, 3) for p in global_xyz]

# Orientation components used for color (map 3/4/5 to RGB)
a = df.actual_TCP_pose_3
b = df.actual_TCP_pose_4
c = df.actual_TCP_pose_5

# Normalize helper to map any vector to [0, 1]
norm01(v) = begin
    vmin, vmax = minimum(v), maximum(v)
    vmax == vmin ? fill(0.5, length(v)) : (v .- vmin) ./ (vmax - vmin)
end

r = norm01(a)
g = norm01(b)
bl = norm01(c)
point_colors = RGBf.(r, g, bl)
points = Point3f.(x, y, z)

fig = Figure()
ax = Axis3(
    fig[1, 1],
    xlabel = "X [m]",
    ylabel = "Y [m]",
    zlabel = "Z [m]",
    title = "TCP Global Task XYZ (color from Rx,Ry,Rz)"
)
scatter!(ax, points; color = point_colors, markersize = 6)

display(GLMakie.Screen(), fig)

GLMakie.Screen(...)

In [4]:
using CSV, DataFrames, GLMakie, Colors

# Define helpers if Cell 1 has not been run in this kernel yet.
if !isdefined(Main, :mat_vec_mul_row)
    function mat_vec_mul_row(v::NTuple{3, Float64}, m::AbstractMatrix{<:Real})
        return (
            v[1] * m[1, 1] + v[2] * m[2, 1] + v[3] * m[3, 1],
            v[1] * m[1, 2] + v[2] * m[2, 2] + v[3] * m[3, 2],
            v[1] * m[1, 3] + v[2] * m[2, 3] + v[3] * m[3, 3],
        )
    end
end

if !isdefined(Main, :base_to_global_task_xyz)
    function base_to_global_task_xyz(base_xyz::NTuple{3, Float64}, arm::Symbol)
        dy_t = 0.225 / 2 + 0.540 / 2
        dz_t = -0.753

        if arm == :left
            dx_t = 0.090 / 2 + 0.010 + 0.110
            r_task_to_base = Float64[
                0.707 0.0 -0.707;
                0.0 -1.0 0.0;
                -0.707 0.0 -0.707;
            ]
        elseif arm == :right
            dx_t = -(0.090 / 2 + 0.010 + 0.110)
            r_task_to_base = Float64[
                0.707 0.0 0.707;
                0.0 -1.0 0.0;
                0.707 0.0 -0.707;
            ]
        else
            error("arm must be :left or :right")
        end

        trans_base_to_task = mat_vec_mul_row((dx_t, dy_t, dz_t), r_task_to_base)
        p_rel = (
            base_xyz[1] - trans_base_to_task[1],
            base_xyz[2] - trans_base_to_task[2],
            base_xyz[3] - trans_base_to_task[3],
        )
        return mat_vec_mul_row(p_rel, transpose(r_task_to_base))
    end
end

# Load both logs
data_left = CSV.read("robot_data_left.csv", DataFrame)
data_right = CSV.read("robot_data_right.csv", DataFrame)

# Arm-frame XYZ
x_left_arm = Float32.(data_left.actual_TCP_pose_0)
y_left_arm = Float32.(data_left.actual_TCP_pose_1)
z_left_arm = Float32.(data_left.actual_TCP_pose_2)

x_right_arm = Float32.(data_right.actual_TCP_pose_0)
y_right_arm = Float32.(data_right.actual_TCP_pose_1)
z_right_arm = Float32.(data_right.actual_TCP_pose_2)

# Convert both to global task frame
left_global = [
    base_to_global_task_xyz((Float64(x_left_arm[i]), Float64(y_left_arm[i]), Float64(z_left_arm[i])), :left)
    for i in eachindex(x_left_arm)
]
right_global = [
    base_to_global_task_xyz((Float64(x_right_arm[i]), Float64(y_right_arm[i]), Float64(z_right_arm[i])), :right)
    for i in eachindex(x_right_arm)
]

x_left = Float32[getindex(p, 1) for p in left_global]
y_left = Float32[getindex(p, 2) for p in left_global]
z_left = Float32[getindex(p, 3) for p in left_global]

x_right = Float32[getindex(p, 1) for p in right_global]
y_right = Float32[getindex(p, 2) for p in right_global]
z_right = Float32[getindex(p, 3) for p in right_global]

points_left = Point3f.(x_left, y_left, z_left)
points_right = Point3f.(x_right, y_right, z_right)

fig = Figure(size = (1100, 800))
ax = Axis3(
    fig[1, 1],
    xlabel = "X [m]",
    ylabel = "Y [m]",
    zlabel = "Z [m]",
    title = "Left + Right TCP (Global Task Frame)"
)

scatter!(ax, points_left; color = RGBf(1.0, 0.0, 0.0), markersize = 5, label = "Left")
scatter!(ax, points_right; color = RGBf(0.0, 0.0, 0.0), markersize = 5, label = "Right")
axislegend(ax, position = :rt)

display(GLMakie.Screen(), fig)

GLMakie.Screen(...)

In [ ]:
import Pkg
for pkg in ["CSV", "DataFrames", "GLMakie", "Colors", "JSON3"]
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

using CSV, DataFrames, GLMakie, Colors, JSON3

# Define transform helpers if prior cells have not run in this kernel yet.
if !isdefined(Main, :mat_vec_mul_row)
    function mat_vec_mul_row(v::NTuple{3, Float64}, m::AbstractMatrix{<:Real})
        return (
            v[1] * m[1, 1] + v[2] * m[2, 1] + v[3] * m[3, 1],
            v[1] * m[1, 2] + v[2] * m[2, 2] + v[3] * m[3, 2],
            v[1] * m[1, 3] + v[2] * m[2, 3] + v[3] * m[3, 3],
        )
    end
end

if !isdefined(Main, :base_to_global_task_xyz)
    function base_to_global_task_xyz(base_xyz::NTuple{3, Float64}, arm::Symbol)
        dy_t = 0.225 / 2 + 0.540 / 2
        dz_t = -0.753

        if arm == :left
            dx_t = 0.090 / 2 + 0.010 + 0.110
            r_task_to_base = Float64[
                0.707 0.0 -0.707;
                0.0 -1.0 0.0;
                -0.707 0.0 -0.707;
            ]
        elseif arm == :right
            dx_t = -(0.090 / 2 + 0.010 + 0.110)
            r_task_to_base = Float64[
                0.707 0.0 0.707;
                0.0 -1.0 0.0;
                0.707 0.0 -0.707;
            ]
        else
            error("arm must be :left or :right")
        end

        trans_base_to_task = mat_vec_mul_row((dx_t, dy_t, dz_t), r_task_to_base)
        p_rel = (
            base_xyz[1] - trans_base_to_task[1],
            base_xyz[2] - trans_base_to_task[2],
            base_xyz[3] - trans_base_to_task[3],
        )
        return mat_vec_mul_row(p_rel, transpose(r_task_to_base))
    end
end

as_string(v) = v === missing ? "" : String(v)

function _try_parse_float(v)
    if v === missing
        return nothing
    end
    try
        return Float64(v)
    catch
        try
            return parse(Float64, String(v))
        catch
            return nothing
        end
    end
end

function _task_graph_candidates(graph)
    tasks = Any[]
    if haskey(graph, :primary_task)
        push!(tasks, graph.primary_task)
    end
    if haskey(graph, :autonomy_queue)
        for t in graph.autonomy_queue
            push!(tasks, t)
        end
    end
    return tasks
end

function _resolve_trace_paths(task_name::String; task_graph_file::Union{Nothing, String}=nothing,
                              left_trace_csv::Union{Nothing, String}=nothing,
                              right_trace_csv::Union{Nothing, String}=nothing)
    left_path = left_trace_csv
    right_path = right_trace_csv

    if (left_path === nothing || right_path === nothing) && task_graph_file !== nothing && isfile(task_graph_file)
        graph = JSON3.read(read(task_graph_file, String))
        for t in _task_graph_candidates(graph)
            name = haskey(t, :name) ? String(t.name) : ""
            if name != task_name
                continue
            end
            if haskey(t, :params)
                params = t.params
                if left_path === nothing && haskey(params, :pose_trace_csv_left)
                    left_path = String(params.pose_trace_csv_left)
                end
                if right_path === nothing && haskey(params, :pose_trace_csv_right)
                    right_path = String(params.pose_trace_csv_right)
                end
            end
            break
        end
    end

    if left_path === nothing
        left_path = "robot_data_left.csv"
    end
    if right_path === nothing
        right_path = "robot_data_right.csv"
    end

    return left_path, right_path
end

function _global_trace_points(df::DataFrame, arm::Symbol; t0::Union{Nothing, Float64}=nothing, t1::Union{Nothing, Float64}=nothing)
    required = ["timestamp", "actual_TCP_pose_0", "actual_TCP_pose_1", "actual_TCP_pose_2"]
    for c in required
        if !(c in names(df))
            error("Trace CSV missing required column: $(c)")
        end
    end

    xs = Float32[]
    ys = Float32[]
    zs = Float32[]

    for row in eachrow(df)
        ts = _try_parse_float(row.timestamp)
        if ts === nothing
            continue
        end
        if t0 !== nothing && ts < t0
            continue
        end
        if t1 !== nothing && ts > t1
            continue
        end

        x = _try_parse_float(row.actual_TCP_pose_0)
        y = _try_parse_float(row.actual_TCP_pose_1)
        z = _try_parse_float(row.actual_TCP_pose_2)
        if x === nothing || y === nothing || z === nothing
            continue
        end

        gx, gy, gz = base_to_global_task_xyz((x, y, z), arm)
        push!(xs, Float32(gx)); push!(ys, Float32(gy)); push!(zs, Float32(gz))
    end

    return Point3f.(xs, ys, zs)
end

function _global_waypoint_points(named_df::DataFrame, rows::Vector{Int}, arm::Symbol)
    xcol = arm == :left ? :left_x : :right_x
    ycol = arm == :left ? :left_y : :right_y
    zcol = arm == :left ? :left_z : :right_z

    pts = Point3f[]
    names_out = String[]

    for i in rows
        x = _try_parse_float(named_df[i, xcol])
        y = _try_parse_float(named_df[i, ycol])
        z = _try_parse_float(named_df[i, zcol])
        if x === nothing || y === nothing || z === nothing
            continue
        end

        gx, gy, gz = base_to_global_task_xyz((x, y, z), arm)
        push!(pts, Point3f(Float32(gx), Float32(gy), Float32(gz)))
        if :waypoint_name in names(named_df)
            push!(names_out, as_string(named_df[i, :waypoint_name]))
        else
            push!(names_out, "")
        end
    end

    return pts, names_out
end

function plot_task(task_name::AbstractString;
                   named_waypoints_csv::String="UR5/waypoints_open_microwave_door.csv",
                   task_graph_file::Union{Nothing, String}=nothing,
                   left_trace_csv::Union{Nothing, String}=nothing,
                   right_trace_csv::Union{Nothing, String}=nothing,
                   markersize_trace::Real=3,
                   markersize_waypoint::Real=13)
    if !isfile(named_waypoints_csv)
        error("named_waypoints_csv not found: $(named_waypoints_csv)")
    end

    named_df = CSV.read(named_waypoints_csv, DataFrame)

    task_rows = if :task_name in names(named_df)
        [i for i in 1:nrow(named_df) if as_string(named_df[i, :task_name]) == task_name]
    else
        collect(1:nrow(named_df))
    end
    if isempty(task_rows)
        if :task_name in names(named_df)
            error("No waypoint rows found for task_name='$(task_name)' in $(named_waypoints_csv)")
        else
            error("No waypoint rows found in $(named_waypoints_csv)")
        end
    end

    # Use waypoint mark times to window the full trace to this task.
    tvals = Float64[]
    if :waypoint_mark_time in names(named_df)
        for i in task_rows
            tv = _try_parse_float(named_df[i, :waypoint_mark_time])
            if tv !== nothing
                push!(tvals, tv)
            end
        end
    end
    t0 = isempty(tvals) ? nothing : minimum(tvals)
    t1 = isempty(tvals) ? nothing : maximum(tvals)

    left_path, right_path = _resolve_trace_paths(String(task_name);
        task_graph_file=task_graph_file,
        left_trace_csv=left_trace_csv,
        right_trace_csv=right_trace_csv,
    )
    if !isfile(left_path)
        error("Left trace CSV not found: $(left_path)")
    end
    if !isfile(right_path)
        error("Right trace CSV not found: $(right_path)")
    end

    left_df = CSV.read(left_path, DataFrame)
    right_df = CSV.read(right_path, DataFrame)

    left_trace_pts = _global_trace_points(left_df, :left; t0=t0, t1=t1)
    right_trace_pts = _global_trace_points(right_df, :right; t0=t0, t1=t1)

    left_wp_pts, left_wp_names = _global_waypoint_points(named_df, task_rows, :left)
    right_wp_pts, right_wp_names = _global_waypoint_points(named_df, task_rows, :right)

    fig = Figure(size=(1250, 900))
    ax = Axis3(
        fig[1, 1],
        xlabel="X [m]",
        ylabel="Y [m]",
        zlabel="Z [m]",
        title="Task: $(task_name) (global task frame)"
    )

    if !isempty(left_trace_pts)
        scatter!(ax, left_trace_pts; color=RGBf(1, 0, 0), markersize=markersize_trace, label="Left trace")
    end
    if !isempty(right_trace_pts)
        scatter!(ax, right_trace_pts; color=RGBf(0, 0, 0), markersize=markersize_trace, label="Right trace")
    end

    if !isempty(left_wp_pts)
        scatter!(ax, left_wp_pts; color=RGBf(1, 0.5, 0.5), marker=:diamond, markersize=markersize_waypoint, label="Left waypoints")
        text!(ax, [p[1] for p in left_wp_pts], [p[2] for p in left_wp_pts], [p[3] for p in left_wp_pts],
              text=[isempty(n) ? "L" : "L:" * n for n in left_wp_names], fontsize=11, color=RGBf(0.7, 0.1, 0.1))
    end
    if !isempty(right_wp_pts)
        scatter!(ax, right_wp_pts; color=RGBf(0.35, 0.35, 0.35), marker=:diamond, markersize=markersize_waypoint, label="Right waypoints")
        text!(ax, [p[1] for p in right_wp_pts], [p[2] for p in right_wp_pts], [p[3] for p in right_wp_pts],
              text=[isempty(n) ? "R" : "R:" * n for n in right_wp_names], fontsize=11, color=RGBf(0.1, 0.1, 0.1))
    end

    axislegend(ax, position=:rt)
    display(GLMakie.Screen(), fig)

    println("Task rows: $(length(task_rows))")
    if !(:task_name in names(named_df))
        println("task_name column not found in waypoint CSV; using all rows from file.")
    end
    println("Trace window: ", t0 === nothing ? "full trace" : "[$(t0), $(t1)]")
    println("Left trace file: $(left_path)")
    println("Right trace file: $(right_path)")

    return fig
end

# Example:
# plot_task("open_microwave_door")  # defaults to UR5/waypoints_open_microwave_door.csv
# plot_task("open_microwave_door"; task_graph_file="UR5/master_task_graph.json")

plot_task (generic function with 1 method)

In [2]:
plot_task("open_microwave_door"; task_graph_file="UR5/master_task_graph.json")

ErrorException: named_waypoints_csv is missing task_name column